In [1]:
print('hello world')

hello world


In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from joblib import Parallel, delayed
import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
import scipy.stats as stats
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, LabelEncoder,PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [2]:
df=pd.read_csv('agronimy mandi dataset.csv')
des_df=pd.read_csv('des dataset.csv')
apy=pd.read_csv('APY.csv')

In [3]:
df

,Sl no.,District Name,Market Name,Commodity,Variety,Grade,Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal),Price Date,State
0,1,Auraiya,Achalda,Wheat,Dara,FAQ,2350.0,2550.0,2450.0,05 Apr 2025,Uttar Pradesh
1,2,Auraiya,Achalda,Wheat,Dara,FAQ,2400.0,2500.0,2470.0,14 Jun 2025,Uttar Pradesh
2,3,Auraiya,Achalda,Wheat,Dara,FAQ,2400.0,2500.0,2470.0,23 Jun 2025,Uttar Pradesh
3,4,Auraiya,Achalda,Wheat,Dara,FAQ,2400.0,2520.0,2470.0,26 Jun 2025,Uttar Pradesh
4,5,Auraiya,Achalda,Wheat,Dara,FAQ,2400.0,2550.0,2500.0,03 Jun 2025,Uttar Pradesh
...,...,...,...,...,...,...,...,...,...,...,...
1118894,17380,Vidisha,Vidisha,Lentil (Masur)(Whole),Organic,FAQ,5200.0,5200.0,5200.0,20 Nov 2024,Madhya Pradesh
1118895,17381,Vidisha,Vidisha,Lentil (Masur)(Whole),Organic,FAQ,5412.0,5412.0,5412.0,27 Feb 2025,Madhya Pradesh
1118896,17382,Vidisha,Vidisha,Lentil (Masur)(Whole),Organic,FAQ,5600.0,5600.0,5600.0,08 Jan 2025,Madhya Pradesh
1118897,17383,Vidisha,Vidisha,Lentil (Masur)(Whole),Organic,FAQ,5640.0,5720.0,5720.0,18 Jan 2025,Madhya Pradesh


In [4]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (1118899, 11)

Columns:
['Sl no.', 'District Name', 'Market Name', 'Commodity', 'Variety', 'Grade', 'Min Price (Rs./Quintal)', 'Max Price (Rs./Quintal)', 'Modal Price (Rs./Quintal)', 'Price Date', 'State']


In [5]:
print(df.dtypes)

print("\nFirst 5 rows:")
print(df.head())

print("\nLast 5 rows:")
print(df.tail())

print("\nPrice Date range:")
print("Minimum:", df["Price Date"].min())
print("Maximum:", df["Price Date"].max())

Sl no.                         int64
District Name                    str
Market Name                      str
Commodity                        str
Variety                          str
Grade                            str
Min Price (Rs./Quintal)      float64
Max Price (Rs./Quintal)      float64
Modal Price (Rs./Quintal)    float64
Price Date                       str
State                            str
dtype: object

First 5 rows:
   Sl no. District Name Market Name Commodity Variety Grade  \
0       1       Auraiya     Achalda     Wheat    Dara   FAQ   
1       2       Auraiya     Achalda     Wheat    Dara   FAQ   
2       3       Auraiya     Achalda     Wheat    Dara   FAQ   
3       4       Auraiya     Achalda     Wheat    Dara   FAQ   
4       5       Auraiya     Achalda     Wheat    Dara   FAQ   

   Min Price (Rs./Quintal)  Max Price (Rs./Quintal)  \
0                   2350.0                   2550.0   
1                   2400.0                   2500.0   
2                   

In [6]:
commodity_counts = df["Commodity"].value_counts()

print(commodity_counts)

Commodity
Brinjal                        111813
Green Chilli                   105141
Bhindi(Ladies Finger)           71929
Mustard                         71572
Wheat                           71305
Cauliflower                     69905
Cabbage                         65487
Soyabean                        57748
Ginger(Green)                   55447
Apple                           54943
Maize                           54256
Garlic                          52747
Carrot                          44283
Banana                          35853
Bajra(Pearl Millet/Cumbu)       31063
Lentil (Masur)(Whole)           29588
Gur(Jaggery)                    29239
Groundnut                       22536
Cotton                          22503
Green Gram (Moong)(Whole)       18972
Arhar (Tur/Red Gram)(Whole)     18586
Mango                           15005
Jowar(Sorghum)                   8978
Name: count, dtype: int64


In [7]:
print("\nTop 10 commodities by number of records:")
print(commodity_counts.head(10))


Top 10 commodities by number of records:
Commodity
Brinjal                  111813
Green Chilli             105141
Bhindi(Ladies Finger)     71929
Mustard                   71572
Wheat                     71305
Cauliflower               69905
Cabbage                   65487
Soyabean                  57748
Ginger(Green)             55447
Apple                     54943
Name: count, dtype: int64


In [8]:
print("\nBottom 10 commodities by number of records:")
print(commodity_counts.tail(10))


Bottom 10 commodities by number of records:
Commodity
Banana                         35853
Bajra(Pearl Millet/Cumbu)      31063
Lentil (Masur)(Whole)          29588
Gur(Jaggery)                   29239
Groundnut                      22536
Cotton                         22503
Green Gram (Moong)(Whole)      18972
Arhar (Tur/Red Gram)(Whole)    18586
Mango                          15005
Jowar(Sorghum)                  8978
Name: count, dtype: int64


In [9]:
# Number of unique dates available for each Commodity + Market
coverage = (
    df.groupby(["Commodity", "Market Name"])["Price Date"]
      .nunique()
      .reset_index(name="unique_days")
      .sort_values("unique_days", ascending=False)
)

print(coverage.head(30).to_string(index=False))

    Commodity         Market Name  unique_days
      Brinjal              Khatra          365
      Brinjal             Kalipur          364
        Wheat  Bishnupur(Bankura)          364
 Green Chilli             Birbhum          364
      Mustard  Bishnupur(Bankura)          364
       Garlic            Gulavati          363
Ginger(Green)            Gulavati          363
      Brinjal             Birbhum          363
 Green Chilli            Gulavati          363
      Brinjal  Bishnupur(Bankura)          363
      Brinjal         Champadanga          361
      Mustard             Asansol          361
      Brinjal             Asansol          361
      Mustard            Durgapur          360
      Cabbage Karsiyang(Matigara)          360
        Wheat            Durgapur          360
      Brinjal            Durgapur          360
  Cauliflower             Sambhal          359
      Brinjal             Sambhal          359
      Cabbage            Durgapur          358
       Cotton

In [10]:
print(apy["Crop_Year"].value_counts().sort_index())

Crop_Year
1997     8548
1998    12289
1999    13011
2000    13588
2001    13305
2002    14178
2003    14660
2004    14149
2005    14062
2006    14675
2007    14680
2008    15149
2009    15339
2010    14888
2011    16126
2012    15278
2013    16178
2014    15586
2015    16339
2016    17416
2017    18008
2018    18301
2019    19264
2020      319
Name: count, dtype: int64


In [12]:
# 2024 demo dataset from original 2018 data
apy_2024_demo = apy[apy["Crop_Year"] == 2018].copy()
apy_2024_demo["Demo_Year"] = 2024

# 2025 demo dataset from original 2019 data
apy_2025_demo = apy[apy["Crop_Year"] == 2019].copy()
apy_2025_demo["Demo_Year"] = 2025

print("2024 Demo:", apy_2024_demo.shape)
print("2025 Demo:", apy_2025_demo.shape)

print("\n2024 Demo year:")
print(apy_2024_demo["Crop_Year"].unique())

print("\n2025 Demo year:")
print(apy_2025_demo["Crop_Year"].unique())

# Save the two demo datasets as separate CSV files

apy_2024_demo.to_csv("2024_demo.csv", index=False)
apy_2025_demo.to_csv("2025_demo.csv", index=False)

print("Files created successfully!")

2024 Demo: (18301, 9)
2025 Demo: (19264, 9)

2024 Demo year:
[2018]

2025 Demo year:
[2019]
Files created successfully!
